# Preliminary modeling using TPM data

## Data loading


Configure root.

In [ ]:
import sys, subprocess
import numpy as np
import pandas as pd
import torch 
import torch.nn as nn
import torch.nn.functional as F
%matplotlib inline
from pathlib import Path

# Configure root
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/singlecell-autoencoder.git"
repo_dir = Path("singlecell-autoencoder")
if COLAB:
    root = Path("/content/singlecell-autoencoder")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])
else:
    root = Path.cwd().parent
sys.path.insert(0, str(root))

# Use GPU if available
torch.manual_seed(111)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Configure bulk data path.

In [ ]:
if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    data_dir = Path("/content/drive/MyDrive/phenotype-prediction-data")
    fcnts_path = str(data_dir / "fcnts_timezero")

else:
    fcnts_path = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/fcnts_timezero"

Load TPM data.

In [ ]:
from src.bulk_data import get_all_tpm_data

# Load data and remove NA genes
bulk_df = get_all_tpm_data(fcnts_path = fcnts_path)

## Train-test split

Try training on all data except for the diagonal, then predict the diagonal?

In [ ]:
# Diagonal mask
mask = (bulk_df["drug1_dose"]) == (bulk_df["drug2_dose"])

# Apply mask
X_train = bulk_df[~mask]
X_test = bulk_df[mask]

# Separate metadata and data
meta = bulk_df.iloc[:, ~bulk_df.columns.str.contains("SP")]
X_train = X_train.iloc[:, X_train.columns.str.contains("SP")]
X_test = X_test.iloc[:, X_test.columns.str.contains("SP")]

# Convert to tensors
X_train = torch.tensor(X_train.to_numpy())
X_test = torch.tensor(X_test.to_numpy())

# Data shape
print(f"# of train samples : {X_train.shape[0]}")
print(f"# of test samples  : {X_test.shape[0]}")

## Training

Training loop.

In [ ]:
from src.model import SimpleAE
from torch.utils.data import Dataset, TensorDataset, DataLoader, random_split
from torch.optim import Adam

def train_simple_ae(
    data: torch.Tensor, 
    batch_size: int,
    epochs: int, 
    lr: float, 
    model_params: dict, 
    device: str,
    train_size = 0.8, 
    seed = 111
):
    """
    Simple training loop for generic AE training
    """

    # Train val split
    generator = torch.Generator().manual_seed(seed)
    dataset = TensorDataset(data)
    train_dataset, val_dataset = random_split(
        dataset = dataset, 
        lengths = [train_size, 1 - train_size], 
        generator = generator
    )
    train_loader = DataLoader(
        dataset = train_dataset, 
        batch_size = batch_size, 
        shuffle = True
    )
    val_loader = DataLoader(
        dataset = val_dataset, 
        batch_size = batch_size, 
        shuffle = True
    )

    model = SimpleAE(**model_params)
    model.to(device)
    model.train()

    optim = Adam(model.parameters(), lr = lr)

    # Store losses
    train_losses = []
    val_losses = []

    for epoch in range(epochs):

        epoch_train_loss = 0.0
        epoch_val_loss = 0.0

        for batch_tuple in train_loader:
            
            # Get batch
            (x,) = batch_tuple
            x = x.to(device).float()

            # Compute loss
            xhat = model(x)
            loss = F.mse_loss(xhat, x)

            # Backprop
            optim.zero_grad()
            loss.backward()
            optim.step()

            # Total loss = avg loss * batchsize
            epoch_train_loss += loss.item() * x.size(0)
        
        # Compute average 
        train_losses.append(epoch_train_loss / len(train_dataset))

        model.eval()

        with torch.no_grad():
            

            for batch_tuple in val_loader:

                (x,) = batch_tuple
                x = x.to(device).float()

                # Compute loss
                xhat = model(x)
                loss = F.mse_loss(xhat, x)

                # Total loss = avg loss * batchsize
                epoch_val_loss += loss.item() * x.size(0)
            
            # Compute average 
            val_losses.append(epoch_val_loss / len(val_dataset))

            model.train()

        # Return loss every 10 
        if (epoch + 1) % 10 == 0:
            print(f"epoch {epoch+1:3d} : Train MSE = {train_losses[-1]:.4f}, Val MSE = {val_losses[-1]:.4f}")
    
    return model, train_losses, val_losses

Train.

In [ ]:
# Model params
num_genes = X_train.shape[1]
input_dim = num_genes
h_dim = 64
latent_dim = 4

# Train params
epochs = 300
lr = 0.001
batch_size = 8

# Simple AE parameters
model_params = {
    "input_dim": input_dim,
    "h_dim": h_dim,
    "latent_dim": latent_dim
}

model, train_losses, val_losses = train_simple_ae(
    data = X_train,
    batch_size = batch_size,
    epochs = epochs,
    lr = lr,
    model_params = model_params,
    device = device
)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(train_losses, label = "Train")
plt.plot(val_losses, label = "Validation")
plt.xlabel("Epochs")
plt.ylabel("MSE loss")
plt.legend()